# Meet TSE server on Colab

Runs the local TSE server on a free Colab GPU and exposes it through a
Cloudflare quick tunnel, so the Chrome extension can use it from a Meet tab.

**Before running:** put this repository and the checkpoint in Google Drive, or
edit `PROJECT_DIR` in the next cell to point at the checkout. The server itself
still binds to `127.0.0.1`; cloudflared is the only public entry point.

The token below is required by `/ws` and `/enroll`. Treat the printed WSS URL
and token as secrets: anyone who has both can use your server and upload
enrollment audio.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Edit these three paths if your Drive layout is different.
PROJECT_DIR = Path('/content/drive/MyDrive/SIC AI Project')
SERVER_DIR = PROJECT_DIR / 'meet-tse' / 'server'
CKPT = PROJECT_DIR / 'src' / 'checkpoints' / 'tse_bsrnn_big_best.pt'
ENROLL = PROJECT_DIR / 'my_voice.wav'   # optional; the popup can upload it instead

assert SERVER_DIR.exists(), f'missing server directory: {SERVER_DIR}'
assert CKPT.exists(), f'missing checkpoint: {CKPT}'
print('server:', SERVER_DIR)
print('checkpoint:', CKPT)
print('enrollment:', ENROLL if ENROLL.exists() else '(upload from popup)')


In [ ]:
import os
os.chdir(SERVER_DIR)
!python -m pip install -q -r requirements.txt
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
import json
import os
import secrets
import subprocess
import sys
import time
import urllib.request

# Keep one token across notebook re-runs. Delete this file to rotate it.
TOKEN_FILE = PROJECT_DIR / '.tse_token'
if TOKEN_FILE.exists() and TOKEN_FILE.read_text().strip():
    TOKEN = TOKEN_FILE.read_text().strip()
else:
    TOKEN = secrets.token_urlsafe(24)
    TOKEN_FILE.write_text(TOKEN)

cmd = [sys.executable, 'server.py', '--ckpt', str(CKPT), '--half', '--token', TOKEN]
if ENROLL.exists():
    cmd += ['--enroll', str(ENROLL)]

log_path = SERVER_DIR / 'server.log'
log_file = open(log_path, 'a', buffering=1)
server_proc = subprocess.Popen(
    cmd, cwd=SERVER_DIR, stdout=log_file, stderr=subprocess.STDOUT,
    start_new_session=True, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
print('starting server pid', server_proc.pid, '(token hidden)')

status = None
for _ in range(180):  # up to 6 minutes: first run may download ECAPA weights
    try:
        with urllib.request.urlopen('http://127.0.0.1:8765/status', timeout=2) as r:
            status = json.load(r)
        if status.get('ok'):
            break
    except Exception:
        pass
    time.sleep(2)
if not status or not status.get('ok'):
    print(log_path.read_text(errors='replace')[-6000:])
    raise RuntimeError('server did not become ready; see server.log above')

print('ready:', {k: status.get(k) for k in ('device', 'enrolled', 'window_s', 'hop_s', 'lookahead_s')})
print('--- server.log tail ---')
print(log_path.read_text(errors='replace')[-2000:])


In [ ]:
import os
import re
import subprocess
import time
import urllib.parse
import urllib.request

CF = Path('/usr/local/bin/cloudflared')
if not CF.exists():
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    urllib.request.urlretrieve(url, CF)
    os.chmod(CF, 0o755)

cf_log_path = SERVER_DIR / 'cloudflared.log'
cf_log = open(cf_log_path, 'w', buffering=1)
cf_proc = subprocess.Popen(
    [str(CF), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8765'],
    stdout=cf_log, stderr=subprocess.STDOUT, start_new_session=True)
print('starting cloudflared pid', cf_proc.pid)

public_url = None
for _ in range(60):
    text = cf_log_path.read_text(errors='replace')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if match:
        public_url = match.group(0)
        break
    time.sleep(1)
if not public_url:
    print(cf_log_path.read_text(errors='replace')[-4000:])
    raise RuntimeError('cloudflared did not print a public URL')

wss_base = public_url.replace('https://', 'wss://') + '/ws'
wss_url = wss_base + '?token=' + urllib.parse.quote(TOKEN)
print('WSS URL (paste into the extension):', wss_url)
print('Token (paste into the extension):', TOKEN)
print('HTTP status:', public_url + '/status')


## Use it

In the extension popup:

1. Set **Server URL** to the printed `WSS URL` (the value already includes
   `?token=...`), or set the base `wss://.../ws` and put the token in the
   **Auth token** field. Both work; the token field overrides a query token.
2. Upload enrollment audio if the server did not start with `--enroll`.
3. Reload the Meet tab so the new socket settings are used.

The first real-time step may take a few seconds while CUDA kernels compile;
watch the server log for `RTF` and the popup for `BYPASS`.


In [ ]:
# Live server log: RTF is printed every 5 s while audio is flowing.
# Interrupt this cell to stop tailing (it does not stop the server).
from IPython import get_ipython
get_ipython().system(f'tail -f "{SERVER_DIR}/server.log"')
